# ES 335: Corn yield + soil + weather pipeline
Builds the tables in `es335_crop/` that notebooks 01-03 use. Runs on Google Colab (downloads ~3,500 CropNet files; slow the first time, cached in Drive after that).

| Output | One row = | Made in |
|---|---|---|
| `soil_clean.csv` | one county | section 1 (ISRIC SoilGrids, 5 km aggregated rasters, mean over the county polygon) |
| `corn_yield.csv` | one county-year, 2016-2022 | section 2 (CropNet USDA corn files) |
| `weather_county_month.csv` | one county-month, Apr-Sep 2017-2022 | section 4 (CropNet WRF-HRRR daily files) |
| `corn_model_table.csv` | one county-year, 2017-2022 | section 5 (inner joins, with a row-count audit) |
| `hrrr_skipped.csv` | one weather file that failed | section 4 |


In [ ]:
!pip install -q rasterio rasterstats geopandas pandas pyarrow huggingface_hub

import os, re, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from huggingface_hub import hf_hub_download, list_repo_files
from google.colab import drive
warnings.filterwarnings("ignore")

drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/es335_crop"      # change if you like
os.makedirs(f"{DATA_DIR}/hrrr_cache", exist_ok=True)

REPO = "CropNet/CropNet"
YEARS_MODEL = list(range(2017, 2023))   # years used for modelling
GROW_MONTHS = [4, 5, 6, 7, 8, 9]        # April-September (assumption for corn)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Soil: download SoilGrids rasters from ISRIC and average per county

In [ ]:
# 1a. County boundaries (contiguous US)
counties = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_20m.zip")
counties["FIPS"] = counties["STATEFP"] + counties["COUNTYFP"]
excluded = ["02", "15", "60", "66", "69", "72", "78"]          # AK, HI, territories
counties = counties[~counties["STATEFP"].isin(excluded)].copy().to_crs("EPSG:4326")
print(len(counties), "contiguous US counties")

# 1b. ISRIC SoilGrids 5 km aggregated rasters
BASE = "https://files.isric.org/soilgrids/latest/data_aggregated/5000m"
soil_layers = {}
for prop in ["nitrogen", "soc", "phh2o"]:
    for depth in ["0-5cm", "5-15cm"]:
        soil_layers[f"{prop}_{depth}"] = f"{BASE}/{prop}/{prop}_{depth}_mean_5000.tif"

# 1c. Download each raster and take the mean over every county polygon
import urllib.request
from rasterstats import zonal_stats
import rasterio

TIF_DIR = f"{DATA_DIR}/soil_tifs"; os.makedirs(TIF_DIR, exist_ok=True)
SOIL_OUT = f"{DATA_DIR}/soil_clean.csv"
FORCE_REDOWNLOAD = False      # True = always call ISRIC again; False = reuse files already in Drive

if os.path.exists(SOIL_OUT) and not FORCE_REDOWNLOAD:
    soil = pd.read_csv(SOIL_OUT, dtype={"FIPS": str})
    print("Reusing soil table already built:", soil.shape)
else:
    soil = counties[["FIPS", "NAME", "STATE_NAME"]].copy()
    for col, url in soil_layers.items():
        tif = f"{TIF_DIR}/{col}.tif"
        if not os.path.exists(tif) or FORCE_REDOWNLOAD:
            print("downloading", col)
            urllib.request.urlretrieve(url, tif)
        with rasterio.open(tif) as src:
            proj = counties.to_crs(src.crs)
            nodata = src.nodata
        stats = zonal_stats(proj, tif, stats="mean", nodata=nodata, all_touched=True)
        soil[col] = [r["mean"] for r in stats]

    # Missing values: fill with the state mean, and record how many counties this affects
    for col in soil_layers:
        print(col, "missing before fill:", soil[col].isna().sum())
        soil[col] = soil.groupby("STATE_NAME")[col].transform(lambda x: x.fillna(x.mean()))

    # Convert integer-scaled SoilGrids units: pH x10, nitrogen cg/kg, SOC dg/kg
    scale = {"phh2o": 10.0, "nitrogen": 100.0, "soc": 10.0}
    for col in soil_layers:
        soil[col] = soil[col] / scale[col.split("_")[0]]

    soil["FIPS"] = soil["FIPS"].str.zfill(5)
    soil.to_csv(SOIL_OUT, index=False)       # cache so later sessions skip the download
    print("Soil table built:", soil.shape)

soil.describe().T

3108 contiguous US counties
Reusing soil table already built: (3108, 9)


,count,mean,std,min,25%,50%,75%,max
nitrogen_0-5cm,3108.0,4.884971,3.079971,0.369051,2.918026,3.838515,5.916953,20.860867
nitrogen_5-15cm,3108.0,2.170799,1.565405,0.327104,1.306833,1.789217,2.484627,18.515434
soc_0-5cm,3108.0,54.677019,37.771744,3.739746,33.236668,44.129751,60.775371,275.320000
soc_5-15cm,3108.0,23.858602,14.053523,3.035771,14.495691,19.746949,28.696116,174.088235
phh2o_0-5cm,3108.0,5.969247,0.812012,4.277974,5.298123,5.890774,6.529805,8.222951
phh2o_5-15cm,3108.0,6.017435,0.821626,4.351982,5.324152,5.912047,6.561389,8.288605


## 2. Corn yield target table (corn only)

In [ ]:
all_files = list_repo_files(REPO, repo_type="dataset")
corn_files = sorted(f for f in all_files if "USDA" in f and "Corn" in f and f.endswith(".csv"))
print(len(corn_files), "corn files:", [os.path.basename(f) for f in corn_files])

raw = pd.concat(
    [pd.read_csv(hf_hub_download(REPO, f, repo_type="dataset")) for f in corn_files],
    ignore_index=True,
)
print(raw.shape)
print(raw.columns.tolist())

# What kinds of rows are in here? Look before filtering.
for c in ["commodity_desc", "agg_level_desc", "domain_desc", "reference_period_desc", "source_desc"]:
    if c in raw.columns:
        print("\n", c); print(raw[c].value_counts(dropna=False).head(10))

7 corn files: ['USDA_Corn_County_2016.csv', 'USDA_Corn_County_2017.csv', 'USDA_Corn_County_2018.csv', 'USDA_Corn_County_2019.csv', 'USDA_Corn_County_2020.csv', 'USDA_Corn_County_2021.csv', 'USDA_Corn_County_2022.csv']
(10246, 14)
['commodity_desc', 'reference_period_desc', 'year', 'state_ansi', 'state_name', 'county_ansi', 'county_name', 'asd_code', 'asd_desc', 'domain_desc', 'source_desc', 'agg_level_desc', 'PRODUCTION, MEASURED IN BU', 'YIELD, MEASURED IN BU / ACRE']

 commodity_desc
commodity_desc
CORN    10246
Name: count, dtype: int64

 agg_level_desc
agg_level_desc
COUNTY    10246
Name: count, dtype: int64

 domain_desc
domain_desc
TOTAL    10246
Name: count, dtype: int64

 reference_period_desc
reference_period_desc
YEAR    10246
Name: count, dtype: int64

 source_desc
source_desc
SURVEY    10246
Name: count, dtype: int64


In [ ]:
Y_COL = "YIELD, MEASURED IN BU / ACRE"

corn = raw.dropna(subset=["state_ansi", "county_ansi"]).copy()
corn["FIPS"] = (
    pd.to_numeric(corn["state_ansi"]).astype(int).astype(str).str.zfill(2)
    + pd.to_numeric(corn["county_ansi"]).astype(int).astype(str).str.zfill(3)
)
corn["Yield"] = pd.to_numeric(corn[Y_COL].astype(str).str.replace(",", ""), errors="coerce")
corn = corn.dropna(subset=["Yield"]).rename(columns={"year": "Year"})

# Duplicate county-years? Inspect BEFORE averaging.
dups = corn.duplicated(["FIPS", "Year"], keep=False)
print("rows involved in duplicates:", dups.sum())
display(corn.loc[dups].sort_values(["FIPS", "Year"]).head(20))

rows involved in duplicates: 0


,commodity_desc,reference_period_desc,Year,state_ansi,state_name,county_ansi,county_name,asd_code,asd_desc,domain_desc,source_desc,agg_level_desc,"PRODUCTION, MEASURED IN BU","YIELD, MEASURED IN BU / ACRE",FIPS,Yield


In [ ]:
# Only run this once you have looked at the duplicates above.
# If they are different categories (e.g. different domain_desc), FILTER instead of averaging.
corn_y = corn.groupby(["FIPS", "Year"], as_index=False)["Yield"].mean()

print(corn_y.groupby("Year").size())            # rows per year (2016 is kept only for the previous-year baseline)
print(corn_y["Yield"].describe())
corn_y.to_csv(f"{DATA_DIR}/corn_yield.csv", index=False)

corn_fips = set(corn_y["FIPS"])
print(len(corn_fips), "counties with at least one corn yield")

Year
2016    1507
2017    1481
2018    1345
2019    1257
2020    1668
2021    1472
2022    1516
dtype: int64
count    10246.000000
mean       157.151464
std         37.482781
min         19.800000
25%        133.900000
50%        162.150000
75%        184.400000
max        270.200000
Name: Yield, dtype: float64
1964 counties with at least one corn yield


## 3. Explore the WRF-HRRR weather files (do this before downloading everything)
Look at the listing, sizes, and format, then narrow `selected` in the next cell.

In [ ]:
hrrr_files = [f for f in all_files if "WRF-HRRR" in f]
print(len(hrrr_files), "files")
print(pd.Series([f.rsplit(".", 1)[-1] for f in hrrr_files]).value_counts())
for f in hrrr_files[:25]:
    print(f)

3529 files
csv         3528
DS_Store       1
Name: count, dtype: int64
WRF-HRRR Computed Dataset/data/2017/AL/HRRR_05_AR_2017-01.csv
WRF-HRRR Computed Dataset/data/2017/AL/HRRR_05_AR_2017-02.csv
WRF-HRRR Computed Dataset/data/2017/AL/HRRR_05_AR_2017-03.csv
WRF-HRRR Computed Dataset/data/2017/AL/HRRR_05_AR_2017-04.csv
WRF-HRRR Computed Dataset/data/2017/AL/HRRR_05_AR_2017-05.csv
WRF-HRRR Computed Dataset/data/2017/AL/HRRR_05_AR_2017-06.csv
WRF-HRRR Computed Dataset/data/2017/AL/HRRR_05_AR_2017-07.csv
WRF-HRRR Computed Dataset/data/2017/AL/HRRR_05_AR_2017-08.csv
WRF-HRRR Computed Dataset/data/2017/AL/HRRR_05_AR_2017-09.csv
WRF-HRRR Computed Dataset/data/2017/AL/HRRR_05_AR_2017-10.csv
WRF-HRRR Computed Dataset/data/2017/AL/HRRR_05_AR_2017-11.csv
WRF-HRRR Computed Dataset/data/2017/AL/HRRR_05_AR_2017-12.csv
WRF-HRRR Computed Dataset/data/2017/AR/HRRR_05_AR_2017-01.csv
WRF-HRRR Computed Dataset/data/2017/AR/HRRR_05_AR_2017-02.csv
WRF-HRRR Computed Dataset/data/2017/AR/HRRR_05_AR_2017-03.csv

In [ ]:
def read_any(path):
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1")

sample_path = hf_hub_download(REPO, hrrr_files[0], repo_type="dataset")   # pick one small file
sample = read_any(sample_path)
print(sample.shape)
print(sample.columns.tolist())
display(sample.head())
if "Daily/Monthly" in sample.columns:
    print(sample["Daily/Monthly"].value_counts())
print(sample["Year"].unique()[:10], sample["Month"].unique()[:12])

(48683, 23)
['Year', 'Month', 'Day', 'Daily/Monthly', 'State', 'County', 'FIPS Code', 'Grid Index', 'Lat (llcrnr)', 'Lon (llcrnr)', 'Lat (urcrnr)', 'Lon (urcrnr)', 'Avg Temperature (K)', 'Max Temperature (K)', 'Min Temperature (K)', 'Precipitation (kg m**-2)', 'Relative Humidity (%)', 'Wind Gust (m s**-1)', 'Wind Speed (m s**-1)', 'U Component of Wind (m s**-1)', 'V Component of Wind (m s**-1)', 'Downward Shortwave Radiation Flux (W m**-2)', 'Vapor Pressure Deficit (kPa)']


,Year,Month,Day,Daily/Monthly,State,County,FIPS Code,Grid Index,Lat (llcrnr),Lon (llcrnr),...,Max Temperature (K),Min Temperature (K),Precipitation (kg m**-2),Relative Humidity (%),Wind Gust (m s**-1),Wind Speed (m s**-1),U Component of Wind (m s**-1),V Component of Wind (m s**-1),Downward Shortwave Radiation Flux (W m**-2),Vapor Pressure Deficit (kPa)
0,2017,1,1.0,Daily,ARKANSAS,ARKANSAS,5001,0.0,34.159311,-91.711795,...,282.594,280.822,0.202,96.6,3.285,3.249,-1.431,-2.724,402.7,0.038
1,2017,1,2.0,Daily,ARKANSAS,ARKANSAS,5001,0.0,34.159311,-91.711795,...,290.327,281.778,32.386,96.8,3.221,3.745,-2.626,-0.033,311.9,0.046
2,2017,1,3.0,Daily,ARKANSAS,ARKANSAS,5001,0.0,34.159311,-91.711795,...,288.150,284.078,0.335,95.8,3.505,3.055,2.428,0.045,233.6,0.061
3,2017,1,4.0,Daily,ARKANSAS,ARKANSAS,5001,0.0,34.159311,-91.711795,...,285.468,272.247,0.254,73.3,7.472,7.812,-0.294,-7.517,2979.4,0.205
4,2017,1,5.0,Daily,ARKANSAS,ARKANSAS,5001,0.0,34.159311,-91.711795,...,275.424,272.067,0.017,76.4,3.854,4.546,-2.013,-3.800,287.5,0.150


Daily/Monthly
Daily      48608
Monthly       75
Name: count, dtype: int64
[2017] [1]


## 4. Aggregate weather to county-month, then county-year
Order matters: average over grid cells per day first, THEN sum precipitation over days (summing across grid cells would inflate it).

In [ ]:
# GROW_MONTHS (April-September) is set in the first cell
HEAT_C = 30.0

HRRR_COLS = {
    "Avg Temperature (K)": "tavg",
    "Max Temperature (K)": "tmax",
    "Min Temperature (K)": "tmin",
    "Precipitation (kg m**-2)": "precip",          # 1 kg m-2 of water = 1 mm
    "Relative Humidity (%)": "rh",
    "Vapor Pressure Deficit (kPa)": "vpd",
    "Downward Shortwave Radiation Flux (W m**-2)": "srad",
}

def county_month_features(w, corn_fips=None, grow_months=GROW_MONTHS):
    """One raw CropNet HRRR file (one state, one month) -> one row per county-month.

    Steps, in this order:
      1. keep daily rows only (each file also has a few 'Monthly' summary rows)
      2. keep growing-season months and counties that ever grew corn
      3. average all grid cells in a county for each day   (spatial mean first)
      4. convert K -> deg C
      5. per month: mean of daily tavg/tmax/tmin/rh/vpd/srad, SUM of daily precip,
         and heat_days = number of days whose county-mean max temperature > 30 C
    Returns None if nothing is left (file outside the season or no corn counties).
    """
    import pandas as pd
    if "Daily/Monthly" in w.columns:
        w = w[w["Daily/Monthly"] == "Daily"]
    w = w[w["Month"].isin(grow_months)].copy()
    if w.empty:
        return None
    w["FIPS"] = pd.to_numeric(w["FIPS Code"], errors="coerce").dropna().astype(int).astype(str).str.zfill(5)
    w = w.dropna(subset=["FIPS"])
    if corn_fips is not None:
        w = w[w["FIPS"].isin(corn_fips)]
    if w.empty:
        return None

    w = w.rename(columns=HRRR_COLS)
    vars_ = list(HRRR_COLS.values())
    daily = w.groupby(["FIPS", "Year", "Month", "Day"], as_index=False)[vars_].mean()
    for c in ["tavg", "tmax", "tmin"]:
        daily[c] = daily[c] - 273.15
    daily["hot"] = (daily["tmax"] > HEAT_C).astype(int)

    out = daily.groupby(["FIPS", "Year", "Month"], as_index=False).agg(
        tavg=("tavg", "mean"), tmax=("tmax", "mean"), tmin=("tmin", "mean"),
        precip=("precip", "sum"), rh=("rh", "mean"), vpd=("vpd", "mean"),
        srad=("srad", "mean"), heat_days=("hot", "sum"),
    )
    out["Year"] = out["Year"].astype(int)
    out["Month"] = out["Month"].astype(int)
    out["heat_days"] = out["heat_days"].astype(float)
    return out


In [ ]:
# Keep only real data files. Narrow further (daily files, years 2017-2022) once you've seen the listing.
DATA_EXT = (".csv", ".parquet")
selected = [f for f in hrrr_files if f.lower().endswith(DATA_EXT)]
print(len(hrrr_files), "->", len(selected), "files after dropping non-data files")

REQUIRED = ["Year", "Month", "Day", "FIPS Code", "Avg Temperature (K)", "Max Temperature (K)",
            "Min Temperature (K)", "Precipitation (kg m**-2)", "Relative Humidity (%)",
            "Vapor Pressure Deficit (kPa)", "Downward Shortwave Radiation Flux (W m**-2)"]

parts, skipped = [], []
for i, f in enumerate(selected):
    cache = f"{DATA_DIR}/hrrr_cache/{re.sub(r'[^A-Za-z0-9]+', '_', f)}.parquet"
    if os.path.exists(cache + ".empty"):          # processed before, no corn counties inside
        continue
    if os.path.exists(cache):
        parts.append(pd.read_parquet(cache)); continue

    path = None
    try:
        path = hf_hub_download(REPO, f, repo_type="dataset", local_dir="/content/hrrr_tmp")
        w = read_any(path)
        missing = [c for c in REQUIRED if c not in w.columns]
        if missing:
            skipped.append((f, f"missing columns: {missing[:3]}")); continue
        out = county_month_features(w, corn_fips)
        if out is None:
            open(cache + ".empty", "w").close()
        else:
            out.to_parquet(cache); parts.append(out)
    except Exception as e:
        skipped.append((f, repr(e)[:150]))
    finally:
        if path and os.path.exists(path):
            os.remove(path)                        # free Colab disk
    if i % 20 == 0: print(f"{i}/{len(selected)} done, skipped so far: {len(skipped)}")

pd.DataFrame(skipped, columns=["file", "reason"]).to_csv(f"{DATA_DIR}/hrrr_skipped.csv", index=False)
print("skipped files:", len(skipped))
for f, r in skipped[:10]: print(" ", f, "->", r)

weather_m = pd.concat(parts, ignore_index=True)
weather_m = weather_m.groupby(["FIPS", "Year", "Month"], as_index=False).mean()
weather_m.to_csv(f"{DATA_DIR}/weather_county_month.csv", index=False)
print(weather_m.shape); weather_m.head()

(download progress output removed to keep the notebook small)
3529 -> 3528 files after dropping non-data files
3500/3528 done, skipped so far: 4
3520/3528 done, skipped so far: 4
skipped files: 4
  WRF-HRRR Computed Dataset/data/2018/NM/HRRR_35_NM_2018-07.csv -> OSError('Consistency check failed: file should be of size 24055772 but has size 24176190 (HRRR_35_NM_2018-07.csv).\nThis is usually due to network iss
  WRF-HRRR Computed Dataset/data/2018/NM/HRRR_35_NM_2018-08.csv -> OSError('Consistency check failed: file should be of size 24016432 but has size 24984029 (HRRR_35_NM_2018-08.csv).\nThis is usually due to network iss
  WRF-HRRR Computed Dataset/data/2018/NM/HRRR_35_NM_2018-09.csv -> OSError('Consistency check failed: file should be of size 23209987 but has size 24870026 (HRRR_35_NM_2018-09.csv).\nThis is usually due to network iss
  WRF-HRRR Computed Dataset/data/2020/NV/HRRR_32_NV_2020-05.csv -> OSError('Consistency check failed: file should be of size 24912926 but has size 368

In [ ]:
# Wide format: one row per county-year, one column per variable-month
feat_cols = ["tavg", "tmax", "tmin", "precip", "rh", "vpd", "srad", "heat_days"]
wide = weather_m.pivot_table(index=["FIPS", "Year"], columns="Month", values=feat_cols)
wide.columns = [f"{v}_m{int(m):02d}" for v, m in wide.columns]
wide = wide.reset_index()

# Season-level summaries
wide["precip_season"] = wide[[c for c in wide if c.startswith("precip_m")]].sum(axis=1, min_count=1)
wide["heat_days_season"] = wide[[c for c in wide if c.startswith("heat_days_m")]].sum(axis=1, min_count=1)
print(wide.shape)

(11715, 52)


## 5. County centroids, then merge and audit

In [ ]:
cent = counties.to_crs(5070).geometry.centroid.to_crs(4326)   # centroid in a projected CRS
geo = pd.DataFrame({"FIPS": counties["FIPS"].values, "STATEFP": counties["STATEFP"].values,
                    "lat": cent.y.values, "lon": cent.x.values})
# Elevation (SRTM) still to add later.

In [ ]:
def audit(name, df, before):
    print(f"{name:<22} rows {before:>6} -> {len(df):>6}   counties {df['FIPS'].nunique()}")

df = corn_y[corn_y["Year"].isin(YEARS_MODEL)].copy()
n0 = len(df); print("corn rows 2017-2022:", n0)

m1 = df.merge(wide, on=["FIPS", "Year"], how="inner");      audit("after weather", m1, n0)
m2 = m1.merge(soil.drop(columns=[c for c in ["NAME", "STATE_NAME"] if c in soil], errors="ignore"),
              on="FIPS", how="inner");                       audit("after soil", m2, len(m1))
m3 = m2.merge(geo, on="FIPS", how="inner");                  audit("after geography", m3, len(m2))

# Which county-years were lost at the weather join?
lost = df.merge(wide[["FIPS", "Year"]], on=["FIPS", "Year"], how="left", indicator=True)
lost = lost[lost["_merge"] == "left_only"]
print("lost at weather join:", len(lost)); print(lost.groupby("Year").size())

print("\nmissing values per column:")
print(m3.isna().sum()[m3.isna().sum() > 0])

m3.to_csv(f"{DATA_DIR}/corn_model_table.csv", index=False)
m3.head()

corn rows 2017-2022: 8739
after weather          rows   8739 ->   8691   counties 1898
after soil             rows   8691 ->   8691   counties 1898
after geography        rows   8691 ->   8691   counties 1898
lost at weather join: 48
Year
2017    37
2018     2
2019     2
2020     3
2021     3
2022     1
dtype: int64

missing values per column:
Series([], dtype: int64)


,FIPS,Year,Yield,heat_days_m04,heat_days_m05,heat_days_m06,heat_days_m07,heat_days_m08,heat_days_m09,precip_m04,...,heat_days_season,nitrogen_0-5cm,nitrogen_5-15cm,soc_0-5cm,soc_5-15cm,phh2o_0-5cm,phh2o_5-15cm,STATEFP,lat,lon
0,01001,2018,165.6,0.0,10.0,25.0,28.0,19.0,22.0,161.103200,...,104.0,2.572278,1.001392,43.375000,14.353750,5.376923,5.356410,01,32.539568,-86.644074
1,01001,2019,165.6,0.0,18.0,21.0,28.0,28.0,30.0,89.377333,...,125.0,2.572278,1.001392,43.375000,14.353750,5.376923,5.356410,01,32.539568,-86.644074
2,01001,2020,156.6,0.0,5.0,12.0,25.0,25.0,13.0,96.603533,...,80.0,2.572278,1.001392,43.375000,14.353750,5.376923,5.356410,01,32.539568,-86.644074
3,01003,2018,152.0,0.0,16.0,24.0,27.0,24.0,24.0,157.759623,...,115.0,7.197011,2.985870,55.498925,19.310215,5.037895,5.072105,01,30.728903,-87.722169
4,01003,2019,150.9,0.0,15.0,24.0,26.0,27.0,30.0,107.608508,...,122.0,7.197011,2.985870,55.498925,19.310215,5.037895,5.072105,01,30.728903,-87.722169


Next: EDA plots, then the split (train 2017-2020, tune 2021, test 2022 + leave-states-out by `STATEFP`), then the county-mean and previous-year baselines (the 2016 rows in `corn_yield.csv` give the previous-year value for 2017).

### Note: very low season rainfall
The lowest April-September rainfall totals (about 1-4 mm) are California counties in 2021, a severe drought year; the highest (about 2,000 mm) are Louisiana/Mississippi in 2017. These are real values, not missing data, so they are kept. Many dry western counties are irrigated, which the model cannot see (see the error analysis on the website).


In [ ]:
# Optional check (Colab): recompute one raw file with county_month_features and compare it
# with the saved weather_county_month.csv. Should print True for every column.
f = "WRF-HRRR Computed Dataset/data/2017/DE/HRRR_10_DE_2017-07.csv"
chk = county_month_features(read_any(hf_hub_download(REPO, f, repo_type="dataset")), corn_fips)
saved = pd.read_csv(f"{DATA_DIR}/weather_county_month.csv", dtype={"FIPS": str})
both = chk.merge(saved, on=["FIPS", "Year", "Month"], suffixes=("_new", "_saved"))
print(len(chk), "rows recomputed,", len(both), "matched in the saved file")
for v in ["tavg", "tmax", "tmin", "precip", "rh", "vpd", "srad", "heat_days"]:
    print(f"{v:<10}", np.allclose(both[f"{v}_new"], both[f"{v}_saved"], rtol=1e-6))
